# retriever를 도구로 전환

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.


## 패키지 설치


In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3 pillow

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("langchain", version("langchain"))
print("langgraph", version("langgraph"))
print("API 키 존재 여부 확인 완료")

langchain 1.4.0
langgraph 1.2.11
API 키 존재 여부 확인 완료


## 로컬 문서와 검색 도구

리트리버(retriever)는 질문을 받아 관련 문서를 반환하는 검색 인터페이스입니다. 이것을 도구로 등록하면 검색 시점과 검색어를 모델이 선택합니다. `search_rules(query: str) -> str`에서 `query`는 모델이 만든 검색 문자열이고, 함수는 `DOCS`를 읽어 관련 문장을 줄바꿈 문자열로 반환합니다. `SEARCH_LOG.append(query)`는 검색 품질을 높이는 기능이 아니라 이번 실행에서 모델이 어떤 검색어를 선택했는지 관찰하는 기록입니다.

고정 RAG는 애플리케이션 코드가 매 질문마다 정해진 검색을 먼저 실행합니다. Agentic RAG는 모델이 검색할지, 어떤 말로 검색할지, 결과가 없을 때 다시 검색할지를 정합니다. `create_agent(..., [search_rules])`는 이 선택권을 제공하지만 검색 결과의 정확성까지 보장하지 않습니다. 도구의 설명과 반환 형식이 모호하면 모델이 적절한 검색어를 만들어도 근거를 잘못 읽을 수 있습니다.


In [3]:
# 읽기 전용 규정과 검색 로그를 선언해 모델의 검색어와 재검색 횟수를 남깁니다.
from langchain.agents import create_agent
from langchain.tools import tool
DOCS = {"대출":"일반 도서는 14일 동안 5권까지 대출할 수 있습니다.", "예약":"예약 도서는 도착 알림 뒤 3일 안에 찾아야 합니다.", "운영":"평일 09:00~20:00 운영합니다."}
SEARCH_LOG=[]
@tool
def search_rules(query: str) -> str:
    """자료실 규정에서 질문과 관련된 문장을 검색합니다."""
    SEARCH_LOG.append(query)
    hits=[f"{k}: {v}" for k,v in DOCS.items() if k in query or any(w in query for w in v.split())]
    return "\n".join(hits) if hits else "검색 결과 없음"

## Agentic RAG 실행


In [4]:
# 예약 질문에서 모델이 search_rules를 선택하고 3일 근거를 답에 사용하는지 확인합니다.
agent=create_agent("openai:gpt-5.4-mini", [search_rules], system_prompt="규정 질문은 search_rules로 검색하세요. 결과가 없으면 다른 검색어로 한 번 재검색하고, 근거 없는 답은 만들지 마세요.")
SEARCH_LOG.clear()
result=agent.invoke({"messages":[{"role":"user","content":"예약한 도서는 며칠 안에 찾아야 하나요?"}]})
print("searches:", SEARCH_LOG)
print("answer:", result["messages"][-1].text)
print("message types:", [type(m).__name__ for m in result["messages"]])
assert SEARCH_LOG and "3" in result["messages"][-1].text

searches: ['예약한 도서는 며칠 안에 찾아야 하나요']
answer: 예약 도서는 **도착 알림을 받은 뒤 3일 안에** 찾아야 합니다.
message types: ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']


## 검색 결과가 없는 질문

상품 지원 시스템에서는 주문 번호가 있는 질문만 주문 도구를 호출하고 일반 안내는 검색 없이 답하게 할 수 있습니다. 반면 모든 질문이 같은 사내 문서 한 묶음에서 답을 찾아야 한다면 고정 검색이 실행 횟수와 재현성을 관리하기 쉽습니다. 모델 선택이 필요한 경우와 애플리케이션 규칙으로 충분한 경우를 구분합니다.

검색 로그가 두 줄이면 재검색했다는 사실은 알 수 있지만 두 번째 검색어가 더 좋았다는 결론은 문서와 답을 함께 읽어야 합니다. 결과 없음 답변도 성공한 거부인지 검색어 실패인지 구분해야 하므로, 이 셀은 검색어와 최종 답을 함께 출력합니다.


In [5]:
# 주차 규정이 없는 질문으로 검색어 변경과 근거 없음 응답을 관찰합니다.
SEARCH_LOG.clear()
missing=agent.invoke({"messages":[{"role":"user","content":"주차 요금 규정을 알려 주세요."}]})
print("searches:", SEARCH_LOG)
print("answer:", missing["messages"][-1].text)
assert SEARCH_LOG

searches: ['주차 요금', '주차료']
answer: 자료실 규정에서 **주차 요금**에 관한 문구를 찾지 못했습니다.  
다른 표현으로도 검색해 봤지만, 관련 규정이 확인되지 않았습니다.

원하시면 제가 대신 다음처럼 다시 찾아볼 수 있습니다:
- **주차비**
- **주차장 이용**
- **차량**
- **요금**
- **사용료**

원하시는 표현이 있으면 알려 주세요.
